In [10]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf

# -----------------------
# Setup assuming dataframe is already loaded in as df
# -----------------------
df["year"] = pd.to_datetime(df["datetime"]).dt.year

PEOPLE = ["Donald Trump", "Joe Biden", "Elon Musk"]
PERSON_LABELS = {
    "Donald Trump": "Trump",
    "Joe Biden": "Biden",
    "Elon Musk": "Musk"
}
MODEL_LABELS = {
    "base": "Base",
    "sent": "Sent",
    "eng": "Eng"
}

# -----------------------
# Helper: run OLS
# -----------------------
def run_ols(formula, data, cluster_col=None):
    model = smf.ols(formula, data=data)
    if cluster_col is not None and cluster_col in data.columns:
        return model.fit(cov_type="cluster", cov_kwds={"groups": data[cluster_col]})
    return model.fit(cov_type="HC1")

# -----------------------
# Horizon columns
# -----------------------
def horizon_cols(df, h):
    if h == "5m":
        return {
            "ret": "ret_5m",
            "vol": "vol_sym_5m",
            "pre_ret": "pre_ret_5m" if "pre_ret_5m" in df.columns else None,
            "pre_vol": "vol_pre_5m" if "vol_pre_5m" in df.columns else None,
        }
    if h == "30m":
        return {
            "ret": "ret_30m",
            "vol": "vol_sym_30m",
            "pre_ret": "pre_ret_30m" if "pre_ret_30m" in df.columns else None,
            "pre_vol": "vol_pre_30m" if "vol_pre_30m" in df.columns else None,
        }
    raise ValueError("h must be '5m' or '30m'")

# -----------------------
# Controls builder
# -----------------------
def build_controls(df, h, include_engagement=True):
    cols = horizon_cols(df, h)
    controls = []

    if include_engagement:
        if "log_retweets" in df.columns:
            controls.append("log_retweets")
        if "log_favorites" in df.columns:
            controls.append("log_favorites")

    if "hour_fe" in df.columns:
        controls.append("C(hour_fe)")
    if "year" in df.columns:
        controls.append("C(year)")

    if cols["pre_ret"] is not None:
        controls.append(cols["pre_ret"])
    if cols["pre_vol"] is not None:
        controls.append(cols["pre_vol"])

    return " + ".join(controls)

# -----------------------
# Formatting helpers
# -----------------------
def stars(p):
    if p < 0.01:
        return "***"
    if p < 0.05:
        return "**"
    if p < 0.10:
        return "*"
    return ""

def format_entry(res, term, decimals=4):
    if term not in res.params.index:
        return ""
    coef = res.params[term]
    se = res.bse[term]
    p = res.pvalues[term]
    return f"{coef:.{decimals}f}{stars(p)} ({se:.{decimals}f})"

# -----------------------
# Model runner
# -----------------------
def run_one_person_model(df, symbol, person, h, depvar, model_type, sentiment_col="overall_sentiment"):
    d = df[(df["symbol"] == symbol) & (df["user_name"] == person)].copy()
    cols = horizon_cols(d, h)
    y = cols["ret"] if depvar == "ret" else cols["vol"]
    cluster_col = "date" if "date" in d.columns else None

    if model_type == "base":
        X = build_controls(d, h, include_engagement=False)
        formula = f"{y} ~ {X}"

    elif model_type == "sent":
        X = build_controls(d, h, include_engagement=True)
        formula = f"{y} ~ {sentiment_col} + {X}"

    elif model_type == "eng":
        X = build_controls(d, h, include_engagement=False)
        formula = f"{y} ~ log_retweets + log_favorites + {X}"

    else:
        raise ValueError("model_type must be one of: 'base', 'sent', 'eng'")

    return run_ols(formula, d, cluster_col=cluster_col)

# -----------------------
# Build long/narrow table
# -----------------------
def build_long_person_table(df, symbol, h, depvar, sentiment_col="overall_sentiment"):
    specs = ["base", "sent", "eng"]
    cols = horizon_cols(df, h)

    row_specs = []
    results = {}

    for person in PEOPLE:
        for spec in specs:
            row_name = f"{PERSON_LABELS[person]} {MODEL_LABELS[spec]}"
            res = run_one_person_model(
                df=df,
                symbol=symbol,
                person=person,
                h=h,
                depvar=depvar,
                model_type=spec,
                sentiment_col=sentiment_col
            )
            results[row_name] = res
            row_specs.append(row_name)

    rows = []
    for row_name in row_specs:
        res = results[row_name]
        row = {"Specification": row_name}

        # coefficients
        row["Log(Retweets + 1)"] = format_entry(res, "log_retweets")
        row["Log(Favorites + 1)"] = format_entry(res, "log_favorites")

        if cols["pre_ret"] is not None:
            row[f"Pre-return ({h})"] = format_entry(res, cols["pre_ret"])
        if cols["pre_vol"] is not None:
            row[f"Pre-vol ({h})"] = format_entry(res, cols["pre_vol"])

        # model stats
        row["Observations"] = int(res.nobs)
        row["$R^2$"] = round(res.rsquared, 4)

        rows.append(row)

    tab = pd.DataFrame(rows)

    # Order columns cleanly
    ordered_cols = ["Specification", "Log(Retweets + 1)", "Log(Favorites + 1)"]
    if cols["pre_ret"] is not None:
        ordered_cols.append(f"Pre-return ({h})")
    if cols["pre_vol"] is not None:
        ordered_cols.append(f"Pre-vol ({h})")
    ordered_cols += ["Observations", "$R^2$"]

    tab = tab[ordered_cols]

    return results, tab

# -----------------------
# Latex export helper
# -----------------------
def print_long_table(df, symbol, h, depvar, sentiment_col="overall_sentiment"):
    results, tab = build_long_person_table(df, symbol, h, depvar, sentiment_col)

    outcome = "returns" if depvar == "ret" else "volatility"
    print(f"\n=== {symbol} | {h} | {outcome} ===")
    display(tab)

    caption = f"{symbol}: Person-specific {h}-minute {outcome} regressions"
    label = f"tab:{symbol.lower()}_{depvar}_{h}_person_long"

    latex_body = tab.to_latex(
        index=False,
        escape=False,
        column_format="l" + "c" * (len(tab.columns) - 1)
    )

    latex_table = f"""
\\begin{{table}}[!htbp]
\\centering
\\caption{{{caption}}}
\\label{{{label}}}
\\scriptsize
\\begin{{adjustbox}}{{max width=\\textwidth}}
{latex_body}
\\end{{adjustbox}}
\\begin{{flushleft}}
\\footnotesize
Notes: Heteroskedasticity-robust standard errors in parentheses. All specifications include year and hour fixed effects; coefficients are omitted for brevity.
\\end{{flushleft}}
\\end{{table}}
"""
    print("\n---------- COPY LATEX BELOW ----------\n")
    print(latex_table)

    return results, tab, latex_table

# -----------------------
# Output the 4 long tables
# -----------------------

# 1) 5m returns
res_ret_5m, tab_ret_5m, latex_ret_5m = print_long_table(
    df, symbol="SPY", h="5m", depvar="ret", sentiment_col="overall_sentiment"
)

# 2) 30m returns
res_ret_30m, tab_ret_30m, latex_ret_30m = print_long_table(
    df, symbol="SPY", h="30m", depvar="ret", sentiment_col="overall_sentiment"
)

# 3) 5m volatility
res_vol_5m, tab_vol_5m, latex_vol_5m = print_long_table(
    df, symbol="SPY", h="5m", depvar="vol", sentiment_col="overall_sentiment"
)

# 4) 30m volatility
res_vol_30m, tab_vol_30m, latex_vol_30m = print_long_table(
    df, symbol="SPY", h="30m", depvar="vol", sentiment_col="overall_sentiment"
)



=== SPY | 5m | returns ===


,Specification,Log(Retweets + 1),Log(Favorites + 1),Pre-return (5m),Pre-vol (5m),Observations,$R^2$
0,Trump Base,,,-0.1891 (0.1674),-135.7210*** (27.5118),150,0.3785
1,Trump Sent,0.0002 (0.0002),0.0000 (0.0000),-0.1554 (0.1592),-137.8678*** (26.5463),150,0.4171
2,Trump Eng,0.0003 (0.0002),0.0000 (0.0000),-0.1601 (0.1567),-135.6396*** (25.6608),150,0.3963
3,Biden Base,,,-0.1265 (0.1789),22.9253 (18.6511),90,0.1272
4,Biden Sent,0.0002 (0.0003),-0.0001 (0.0003),-0.1510 (0.1535),18.2763 (17.8247),90,0.1680
5,Biden Eng,0.0001 (0.0003),0.0001 (0.0003),-0.1254 (0.1741),22.6402 (18.4919),90,0.1397
6,Musk Base,,,-0.0649 (0.1170),10.7260 (68.3612),81,0.0794
7,Musk Sent,0.0004* (0.0002),-0.0005** (0.0003),-0.0357 (0.1191),-6.6544 (81.7336),81,0.1624
8,Musk Eng,0.0002 (0.0002),-0.0004 (0.0002),-0.0537 (0.1017),18.4323 (62.1299),81,0.1271



---------- COPY LATEX BELOW ----------


\begin{table}[!htbp]
\centering
\caption{SPY: Person-specific 5m-minute returns regressions}
\label{tab:spy_ret_5m_person_long}
\scriptsize
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccccc}
\toprule
Specification & Log(Retweets + 1) & Log(Favorites + 1) & Pre-return (5m) & Pre-vol (5m) & Observations & $R^2$ \\
\midrule
Trump Base &  &  & -0.1891 (0.1674) & -135.7210*** (27.5118) & 150 & 0.378500 \\
Trump Sent & 0.0002 (0.0002) & 0.0000 (0.0000) & -0.1554 (0.1592) & -137.8678*** (26.5463) & 150 & 0.417100 \\
Trump Eng & 0.0003 (0.0002) & 0.0000 (0.0000) & -0.1601 (0.1567) & -135.6396*** (25.6608) & 150 & 0.396300 \\
Biden Base &  &  & -0.1265 (0.1789) & 22.9253 (18.6511) & 90 & 0.127200 \\
Biden Sent & 0.0002 (0.0003) & -0.0001 (0.0003) & -0.1510 (0.1535) & 18.2763 (17.8247) & 90 & 0.168000 \\
Biden Eng & 0.0001 (0.0003) & 0.0001 (0.0003) & -0.1254 (0.1741) & 22.6402 (18.4919) & 90 & 0.139700 \\
Musk Base &  &  & -0.0649 (0.1170)

,Specification,Log(Retweets + 1),Log(Favorites + 1),Pre-return (30m),Pre-vol (30m),Observations,$R^2$
0,Trump Base,,,-0.0824 (0.2454),-23.5545 (60.6333),150,0.0585
1,Trump Sent,0.0001 (0.0003),0.0000 (0.0000),-0.0810 (0.2460),-25.7354 (61.7102),150,0.0836
2,Trump Eng,0.0001 (0.0003),0.0000 (0.0000),-0.0705 (0.2502),-24.8516 (60.7129),150,0.0665
3,Biden Base,,,0.7047 (0.7412),73.6219 (57.2651),90,0.0748
4,Biden Sent,0.0013 (0.0016),-0.0018 (0.0015),0.6520 (0.6293),83.0926 (64.6198),90,0.1708
5,Biden Eng,0.0020 (0.0022),-0.0025 (0.0022),0.6957 (0.7335),73.0401 (56.6952),90,0.0903
6,Musk Base,,,0.1583 (0.1960),-88.6540 (86.9939),81,0.1067
7,Musk Sent,0.0009 (0.0007),-0.0007 (0.0008),0.0777 (0.1792),-93.6736 (86.8015),81,0.1713
8,Musk Eng,0.0010* (0.0006),-0.0008 (0.0006),0.0706 (0.1714),-84.9824 (86.7447),81,0.1391



---------- COPY LATEX BELOW ----------


\begin{table}[!htbp]
\centering
\caption{SPY: Person-specific 30m-minute returns regressions}
\label{tab:spy_ret_30m_person_long}
\scriptsize
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccccc}
\toprule
Specification & Log(Retweets + 1) & Log(Favorites + 1) & Pre-return (30m) & Pre-vol (30m) & Observations & $R^2$ \\
\midrule
Trump Base &  &  & -0.0824 (0.2454) & -23.5545 (60.6333) & 150 & 0.058500 \\
Trump Sent & 0.0001 (0.0003) & 0.0000 (0.0000) & -0.0810 (0.2460) & -25.7354 (61.7102) & 150 & 0.083600 \\
Trump Eng & 0.0001 (0.0003) & 0.0000 (0.0000) & -0.0705 (0.2502) & -24.8516 (60.7129) & 150 & 0.066500 \\
Biden Base &  &  & 0.7047 (0.7412) & 73.6219 (57.2651) & 90 & 0.074800 \\
Biden Sent & 0.0013 (0.0016) & -0.0018 (0.0015) & 0.6520 (0.6293) & 83.0926 (64.6198) & 90 & 0.170800 \\
Biden Eng & 0.0020 (0.0022) & -0.0025 (0.0022) & 0.6957 (0.7335) & 73.0401 (56.6952) & 90 & 0.090300 \\
Musk Base &  &  & 0.1583 (0.1960) & -88.6540

,Specification,Log(Retweets + 1),Log(Favorites + 1),Pre-return (5m),Pre-vol (5m),Observations,$R^2$
0,Trump Base,,,0.0020 (0.0021),3.8448*** (0.3785),150,0.9215
1,Trump Sent,-0.0000 (0.0000),-0.0000 (0.0000),0.0018 (0.0020),3.8695*** (0.3685),150,0.9254
2,Trump Eng,-0.0000 (0.0000),-0.0000 (0.0000),0.0018 (0.0020),3.8681*** (0.3664),150,0.9253
3,Biden Base,,,0.0046 (0.0031),0.7746** (0.3158),90,0.5849
4,Biden Sent,-0.0000 (0.0000),0.0000 (0.0000),0.0040 (0.0026),0.7069** (0.3107),90,0.6045
5,Biden Eng,-0.0000 (0.0000),0.0000 (0.0000),0.0046 (0.0030),0.7794** (0.3211),90,0.5860
6,Musk Base,,,0.0007** (0.0003),0.4037** (0.1984),81,0.1497
7,Musk Sent,0.0000** (0.0000),-0.0000** (0.0000),0.0007* (0.0004),0.1567 (0.2859),81,0.2701
8,Musk Eng,0.0000* (0.0000),-0.0000* (0.0000),0.0007** (0.0003),0.3453 (0.2232),81,0.1826



---------- COPY LATEX BELOW ----------


\begin{table}[!htbp]
\centering
\caption{SPY: Person-specific 5m-minute volatility regressions}
\label{tab:spy_vol_5m_person_long}
\scriptsize
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccccc}
\toprule
Specification & Log(Retweets + 1) & Log(Favorites + 1) & Pre-return (5m) & Pre-vol (5m) & Observations & $R^2$ \\
\midrule
Trump Base &  &  & 0.0020 (0.0021) & 3.8448*** (0.3785) & 150 & 0.921500 \\
Trump Sent & -0.0000 (0.0000) & -0.0000 (0.0000) & 0.0018 (0.0020) & 3.8695*** (0.3685) & 150 & 0.925400 \\
Trump Eng & -0.0000 (0.0000) & -0.0000 (0.0000) & 0.0018 (0.0020) & 3.8681*** (0.3664) & 150 & 0.925300 \\
Biden Base &  &  & 0.0046 (0.0031) & 0.7746** (0.3158) & 90 & 0.584900 \\
Biden Sent & -0.0000 (0.0000) & 0.0000 (0.0000) & 0.0040 (0.0026) & 0.7069** (0.3107) & 90 & 0.604500 \\
Biden Eng & -0.0000 (0.0000) & 0.0000 (0.0000) & 0.0046 (0.0030) & 0.7794** (0.3211) & 90 & 0.586000 \\
Musk Base &  &  & 0.0007** (0.0003) & 0.4037

,Specification,Log(Retweets + 1),Log(Favorites + 1),Pre-return (30m),Pre-vol (30m),Observations,$R^2$
0,Trump Base,,,-0.0013 (0.0011),1.3620* (0.7978),150,0.5617
1,Trump Sent,0.0000 (0.0000),-0.0000 (0.0000),-0.0014 (0.0012),1.3745* (0.8024),150,0.5663
2,Trump Eng,0.0000* (0.0000),-0.0000 (0.0000),-0.0013 (0.0011),1.3749* (0.8005),150,0.5655
3,Biden Base,,,-0.0494 (0.0474),-3.2081 (3.6193),90,0.0781
4,Biden Sent,-0.0000 (0.0001),0.0000 (0.0001),-0.0448 (0.0401),-3.9044 (4.0917),90,0.1995
5,Biden Eng,-0.0001 (0.0001),0.0001 (0.0001),-0.0487 (0.0474),-3.1470 (3.6269),90,0.0839
6,Musk Base,,,0.0001 (0.0013),1.6010*** (0.4617),81,0.6064
7,Musk Sent,-0.0000 (0.0000),-0.0000 (0.0000),0.0004 (0.0012),1.5225*** (0.4419),81,0.6409
8,Musk Eng,0.0000 (0.0000),-0.0000 (0.0000),0.0004 (0.0012),1.5561*** (0.4647),81,0.6198



---------- COPY LATEX BELOW ----------


\begin{table}[!htbp]
\centering
\caption{SPY: Person-specific 30m-minute volatility regressions}
\label{tab:spy_vol_30m_person_long}
\scriptsize
\begin{adjustbox}{max width=\textwidth}
\begin{tabular}{lcccccc}
\toprule
Specification & Log(Retweets + 1) & Log(Favorites + 1) & Pre-return (30m) & Pre-vol (30m) & Observations & $R^2$ \\
\midrule
Trump Base &  &  & -0.0013 (0.0011) & 1.3620* (0.7978) & 150 & 0.561700 \\
Trump Sent & 0.0000 (0.0000) & -0.0000 (0.0000) & -0.0014 (0.0012) & 1.3745* (0.8024) & 150 & 0.566300 \\
Trump Eng & 0.0000* (0.0000) & -0.0000 (0.0000) & -0.0013 (0.0011) & 1.3749* (0.8005) & 150 & 0.565500 \\
Biden Base &  &  & -0.0494 (0.0474) & -3.2081 (3.6193) & 90 & 0.078100 \\
Biden Sent & -0.0000 (0.0001) & 0.0000 (0.0001) & -0.0448 (0.0401) & -3.9044 (4.0917) & 90 & 0.199500 \\
Biden Eng & -0.0001 (0.0001) & 0.0001 (0.0001) & -0.0487 (0.0474) & -3.1470 (3.6269) & 90 & 0.083900 \\
Musk Base &  &  & 0.0001 (0.0013) & 1.6010**